# Upload a CSV file and resume an interrupted transfer

SDK 0.4 source candidate. The default no-network fixture deliberately loses one chunk acknowledgement after accepting it. Explicit live mode uploads this tiny synthetic CSV to a separately accepted service; a successful live transfer does not imply an interruption occurred. Created uploads/datasets are retained. No fit, prediction, deletion or server cancellation occurs.

In [ ]:
from pathlib import Path
import tempfile
from welt import TransportError
from support import notebook_file_client, notebook_async_file_client

schema = {"columns": ["customer_code", "amount", "flag", "target"],
          "types": ["string", "number", "boolean", "number"]}
source_bytes = b"customer_code,amount,flag,target\n0012,1.5,true,0\n0042,,false,1\n0012,2.5,true,0\n"

The explicit schema preserves leading-zero identifiers, finite numeric values, booleans and empty-field nulls. Keep an unchanged source file, target/schema declaration and the upload ID after an acknowledged interruption. The SDK rehashes all bytes, including chunks already received.

In [ ]:
with tempfile.TemporaryDirectory(prefix="welt-file-example-") as folder:
    path = Path(folder) / "training.csv"
    path.write_bytes(source_bytes)
    with notebook_file_client() as client:
        upload_id = None
        try:
            dataset = client.upload_file(path, target="target", schema=schema)
        except TransportError as error:
            upload_id = getattr(error, "upload_id", None)
            if upload_id is None:
                raise  # No acknowledged identity is available for safe resume.
            dataset = client.resume_upload(upload_id, path, target="target", schema=schema)
        assert dataset["columns"] == schema["columns"]
        assert dataset["types"] == ["categorical", "numeric", "categorical", "numeric"]
        assert dataset["rows"] == 3
        reopened = client.dataset(dataset["id"])
        assert reopened["id"] == dataset["id"]
        if upload_id is not None:
            info = client.upload_info(upload_id)
            assert info["status"] == "complete"
            assert info["dataset_id"] == dataset["id"]
            repeated = client.resume_upload(upload_id, path, target="target", schema=schema)
            assert repeated["id"] == dataset["id"]
    async with notebook_async_file_client() as client:
        async_dataset = await client.upload_file(path, target="target", schema=schema)
        assert async_dataset["columns"] == schema["columns"]
        assert async_dataset["types"] == ["categorical", "numeric", "categorical", "numeric"]
        assert async_dataset["rows"] == 3

The dataset ID can be selected explicitly for a supported fit; target-free uploads can be selected for discovery. Upload success does not expand a model envelope. Preview defaults are 1 MiB chunks, 64 MiB raw file, four active uploads/workspace and 24-hour logical expiry, with the original normalized dataset/model limits. Cancelling an async await leaves any accepted upload resumable. Parquet transport, batch and physical cleanup are separate gates. Fixture execution validates SDK plumbing, not backend isolation, hosted capacity or model behavior.